In [ ]:
from huggingface_hub import login
import os
import sys
import csv
from tqdm import trange
from transformers import AutoModel,AutoTokenizer
FILE_PATH = './QA_results_GT.csv'
os.environ["OPENAI_API_KEY"] = "your key"
os.environ["OPENAI_API_BASE"] = "https://api.openai.com/v1"
import pandas as pd
import os

In [ ]:
# set directory
data_dir = r"your dire"

In [ ]:
ANA_FILE_PATH = 'your dire/Default_output.csv'

# Only keep: Question, Gold Answer, MiniRAG answer
minianswer_LIST = []
QUESTION_LIST = []
GA_LIST = []
filelength = 0
with open(ANA_FILE_PATH, mode='r', encoding='utf-8') as question_file:
    reader = csv.DictReader(question_file)
    for row in reader:
        QUESTION_LIST.append(row['Question'])
        GA_LIST.append(row['Gold Answer'])
        # Comment out other systems; only keep MiniRAG
        # naiveanswer_LIST.append(row['naive'])
        # lightraganswer_LIST.append(row['lightrag'])
        minianswer_LIST.append(row['minirag'])
        filelength = filelength+1

In [3]:
PROMPT = """
Now, I'll give you a question, a gold answer to this question, and one answer provided by a student.

Determine the answer according to the following rules:
If the answer is correct, get 1 point.
If the answer is irrelevant to the question, it will receive 0 points.
If the answer is incorrect, get -1 point.

Return your answer in JSON mode. Use the key "Score".

For example:

Question:
When does Li Hua arrive to the city?

Gold Answer:
20260105

Answer: LiHua arrived on the afternoon of January 5th

output:
{{
"Score": 1
}}


Real data:

Question:
{question}
Gold Answer:
{ga}

Answer: {mini}

output:

"""

In [7]:
# openai via custom proxy gateway (set OPENAI_BASE_URL and OPENAI_API_KEY in env)
from openai import OpenAI
from tqdm import trange
import os, time

BASE_URL = os.getenv("OPENAI_BASE_URL", "https://api.openai.com/v1")  # your proxy, override via env
API_KEY = os.getenv("OPENAI_API_KEY")  # set this in your shell

chatbot = OpenAI(api_key=API_KEY, base_url=BASE_URL)

def call_with_retry(prompt: str, retries: int = 5, backoff: float = 1.5) -> str:
    for i in range(retries):
        try:
            resp = chatbot.chat.completions.create(
                messages=[{"role": "system", "content": prompt}],
                model="gpt-4o",
            )
            return resp.choices[0].message.content.strip()
        except Exception as e:
            if i == retries - 1:
                raise
            time.sleep(backoff * (2 ** i))

chat_list = []
for i in trange(filelength):
    p = PROMPT.format(question=QUESTION_LIST[i], ga=GA_LIST[i], mini=minianswer_LIST[i])
    chat_list.append(call_with_retry(p))


100%|██████████| 86/86 [01:07<00:00,  1.27it/s]


In [8]:
import json
import json_repair

# Parse LLM outputs
chat_score_list = []
for chat in chat_list:
    try:
        data = json_repair.loads(chat.strip('```json').strip('```'))
        chat_score_list.append(data)
    except Exception:
        chat_score_list.append({"Score": 0})
        print('Error in chat:', chat)

# Only one score now
all_scores = [data.get('Score', 0) for data in chat_score_list]

num_all = len(all_scores)
num_pos = all_scores.count(1)
num_zero = all_scores.count(0)
num_neg = all_scores.count(-1)

print(f"MiniRAG Score 1: {num_pos / num_all * 100:.2f}\%, Score 0: {num_zero / num_all * 100:.2f}\%, Score -1: {num_neg / num_all * 100:.2f}\%")

MiniRAG Score 1: 68.60\%, Score 0: 13.95\%, Score -1: 17.44\%


In [10]:
import pandas as pd

# Map numeric score to human-readable label
def score_to_label(score):
    if score == 1:
        return "Correct"
    elif score == 0:
        return "Unclear"
    elif score == -1:
        return "Wrong"
    else:
        return "Invalid"

# Build output DataFrame
df_out = pd.DataFrame({
    "Question": QUESTION_LIST,
    "GroundTruth": GA_LIST,
    "MiniAnswer": minianswer_LIST,
    "QWen_Output": chat_list,
    "Score": all_scores,
})

# Add correctness label
df_out["Correctness"] = df_out["Score"].apply(score_to_label)

# Save to CSV
df_out.to_csv("Default_output.csv", index=False)

print("Saved to Default_output.csv with Correctness column.")


Saved to Default_output.csv with Correctness column.


In [ ]:
# read Default_output.xlsx
Default_output = pd.read_excel(os.path.join(data_dir, 'Default_output.xlsx'))
Default_output

,Question,GroundTruth,MiniAnswer,Correctness,Evidence,Type,ques_reason,Origin_evalu
0,Did Yuriko ask Li Hua for help with her studio...,Yes,"Yes, Yuriko did indeed ask Li Hua for help reg...",Correct,20260223_15:00<and>20260225_15:00,Multi,NaN,NaN
1,"Who does Li Hua go to watch the movie ""Overwat...",Wolfgang,"Based on the conversation provided, WolfgangSc...",Correct,20260122_17:00<and>20260121_13:00,Multi,NaN,NaN
2,Who wished Li Hua a happy Lunar New Year?,Adam Smith & Jennifer Moore & Wolfgang Schulz,"In the conversation, two individuals wished Li...",Correct,20260119_11:30<and>20260119_14:30<and>20260119...,Multi,NaN,NaN
3,Who introduced the bread delivery service and ...,HaileyJohnson,The bread delivery service was initially intro...,Correct,20260318_15:00<and>20260329_13:00,Multi,NaN,NaN
4,What was the content of the first-ever deliver...,a fresh sourdough loaf and a bottle of milk an...,The first-ever delivery from Hailey to LiHua w...,Correct,20260317_08:00<and>20260318_15:00,Multi,NaN,NaN
...,...,...,...,...,...,...,...,...
81,When did Adam Smith inform Li Hua about potent...,20260301_13:00,"According to the conversation, Adam Smith info...",Correct,20260301_13:00,Single,NaN,Wrong
82,When did Li Hua inform Adam Smith that the ren...,20260301_10:00,"According to the messages provided, Li Hua inf...",Wrong,20260301_10:00,Single,multiple time information,NaN
83,What new feature does Yuriko Yamamoto consider...,A blog section,"Based on the conversation, Yuriko Yamamoto con...",Wrong,20260307_13:00,Single,key information lost,NaN
84,What does Li Hua suggest to Hailey regarding t...,Twice a week on Mondays and Fridays at 8am,Based on the conversation between Li Hua and H...,Wrong,20260318_15:00,Single,"time question, Can not give precise informatio...",NaN


In [8]:
# show how many Correctness are not 'Correct' by Type
print(Default_output[Default_output['Correctness'] != 'Correct']['Type'].value_counts())
# show the distribution of Type
print(Default_output['Type'].value_counts())

# show the percentile of Single Type questions are not Correct and how many Multi Type questions are not Correct
single_incorrect = Default_output[(Default_output['Type'] == 'Single') & (Default_output['Correctness'] != 'Correct')]
multi_incorrect = Default_output[(Default_output['Type'] == 'Multi') & (Default_output['Correctness'] != 'Correct')]
print(f"Percentile of Single Type questions not Correct: {len(single_incorrect)/len(Default_output[Default_output['Type'] == 'Single'])*100:.2f}%")
print(f"Percentile of Multi Type questions not Correct: {len(multi_incorrect)/len(Default_output[Default_output['Type'] == 'Multi'])*100:.2f}%")

Type
Single    17
Multi      5
Name: count, dtype: int64
Type
Single    69
Multi     17
Name: count, dtype: int64
Percentile of Single Type questions not Correct: 24.64%
Percentile of Multi Type questions not Correct: 29.41%


In [14]:
# show the percentile distribution of Correctness
print(Default_output['Correctness'].value_counts(normalize=True) * 100)
# merge Correctness value to Origin_evalu if Origin_evalu is NaN
Default_output['Origin_evalu'] = Default_output['Origin_evalu'].fillna(Default_output['Correctness'])
# show the percentile distribution of Origin_evalu
print(Default_output['Origin_evalu'].value_counts(normalize=True) * 100)

Correctness
Correct    74.418605
Wrong      22.093023
Unclear     3.488372
Name: proportion, dtype: float64
Origin_evalu
Correct    68.604651
Wrong      17.441860
Unclear    13.953488
Name: proportion, dtype: float64


In [ ]:
# show the value distribution of ques_reason and to a df
ques_reason_df = Default_output['ques_reason'].value_counts().reset_index()
ques_reason_df.columns = ['ques_reason', 'count']
ques_reason_df

,ques_reason,count
0,Can not distinguish between when and where and...,4
1,time question,4
2,Can not give precise information extraction. V...,3
3,multiple time information,3
4,multiple document,2
5,"Interaction between people, not time sensitive",1
6,first time question\n\nbut the document is abo...,1
7,key information lost,1
8,"time question, Can not give precise informatio...",1
9,Lost in multiple time information,1
